In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
proj_root = Path.cwd().parent
sys.path.insert(0, str(proj_root))

from models.crack_classifier import build_model
from data.impedance_dataset import build_transforms

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seeds
def seed_all(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Impedance CNN Training Notebook

Train a Convolutional Neural Network on Eddy Current Testing impedance plane graphs for binary crack classification.

**Model**: SmallCNN or ResNet18  
**Task**: Binary classification (Crack=1, No Crack=0)  
**Input**: Grayscale 224×224 impedance plane images  
**Output**: 2-class logits (converted to probabilities)

## 1. Data Loading & Preparation

Create synthetic impedance dataset for demo, or load from CSV with real data.

In [ ]:
# Configuration
DATA_PATH = None  # Set to "data/metadata.csv" to load real data
USE_SYNTHETIC = True
NUM_SAMPLES = 400
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 2

if USE_SYNTHETIC or DATA_PATH is None:
    print("Creating synthetic impedance dataset...")
    
    # Synthetic data: random 224x224 grayscale images
    X = torch.randn(NUM_SAMPLES, 1, IMG_SIZE, IMG_SIZE)
    y = torch.randint(0, NUM_CLASSES, (NUM_SAMPLES,))
    
    # Split: 60% train, 20% val, 20% test
    train_size = int(0.6 * NUM_SAMPLES)
    val_size = int(0.2 * NUM_SAMPLES)
    
    X_train, y_train = X[:train_size], y[:train_size]
    X_val, y_val = X[train_size:train_size+val_size], y[train_size:train_size+val_size]
    X_test, y_test = X[train_size+val_size:], y[train_size+val_size:]
    
    print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    print(f"Class distribution (train):")
    unique, counts = torch.unique(y_train, return_counts=True)
    for cls, cnt in zip(unique, counts):
        print(f"  Class {cls.item()}: {cnt.item()}")
else:
    print(f"Loading data from {DATA_PATH}")
    # TODO: Implement CSV loading with CrackDataset
    raise NotImplementedError("CSV loading not yet implemented")

# Create datasets and dataloaders
ds_train = TensorDataset(X_train, y_train)
ds_val = TensorDataset(X_val, y_val)
ds_test = TensorDataset(X_test, y_test)

dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True)
dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False)
dl_test = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False)

print("\nDataloaders created successfully!")

## 2. Data Visualization

Visualize sample impedance graphs and label distribution.

In [ ]:
# Visualize sample impedance graphs
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    idx = np.random.randint(0, len(X_train))
    img = X_train[idx].squeeze().numpy()
    label = y_train[idx].item()
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Label: {label}\n({'Crack' if label == 1 else 'No Crack'})")
    ax.axis('off')
plt.tight_layout()
plt.show()

# Class distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, (name, data) in enumerate([('Train', y_train), ('Val', y_val), ('Test', y_test)]):
    unique, counts = torch.unique(data, return_counts=True)
    ax = axes[idx]
    ax.bar([f"Class {c.item()}" for c in unique], counts.numpy())
    ax.set_title(f"{name} Set Class Distribution")
    ax.set_ylabel("Count")
    for i, v in enumerate(counts):
        ax.text(i, v, str(v.item()), ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 3. Model Selection & Training Setup

Choose between SmallCNN and ResNet18, then train the model.

In [ ]:
# Model configuration
ARCH = "small_cnn"  # or "resnet18"
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4

# Build model
model = build_model(ARCH, num_classes=NUM_CLASSES, pretrained=False, dropout=0.2, grayscale=True).to(device)
print(f"Model: {type(model).__name__}")
print(f"Architecture: {ARCH}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

# History tracking
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'epochs': []
}

## 4. Training Loop with Live Metrics

Train the model with epoch-by-epoch progress tracking.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss = criterion(logits, labels)
            running_loss += loss.item() * imgs.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

# Training loop
best_val_acc = 0.0
pbar = tqdm(range(1, EPOCHS + 1), desc="Training")

for epoch in pbar:
    train_loss, train_acc = train_epoch(model, dl_train, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(model, dl_val, criterion, device)
    scheduler.step()
    
    history['epochs'].append(epoch)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    pbar.set_postfix({
        'train_loss': f'{train_loss:.4f}',
        'train_acc': f'{train_acc:.4f}',
        'val_loss': f'{val_loss:.4f}',
        'val_acc': f'{val_acc:.4f}'
    })
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # Save best model
        torch.save(model.state_dict(), 'best_model.pt')

print(f"\nTraining complete! Best val_acc: {best_val_acc:.4f}")

## 5. Training History Visualization

Plot loss and accuracy curves.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
axes[0].plot(history['epochs'], history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['epochs'], history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy curves
axes[1].plot(history['epochs'], history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['epochs'], history['val_acc'], label='Val Acc', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 6. Model Evaluation on Test Set

Evaluate on test data with confusion matrix, ROC curve, and classification report.

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

# Get predictions on test set
y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for imgs, labels in dl_test:
        imgs = imgs.to(device)
        logits = model(imgs)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(labels.numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        y_prob.extend(probs[:, 1].cpu().numpy().tolist())  # Probability of class 1 (crack)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

# Test metrics
test_loss, test_acc = validate_epoch(model, dl_test, criterion, device)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["No Crack", "Crack"]))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(cm)

## 7. Confusion Matrix & ROC Curve

Visualize model performance metrics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['No Crack', 'Crack'], yticklabels=['No Crack', 'Crack'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})', color='blue', lw=2)
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"ROC AUC: {roc_auc:.4f}")

## 8. Sample Predictions Visualization

Show test samples with predictions and confidence.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
for i, ax in enumerate(axes.flat):
    idx = i % len(X_test)
    img = X_test[idx].squeeze().numpy()
    true_label = y_true[idx]
    pred_label = y_pred[idx]
    confidence = y_prob[idx] if pred_label == 1 else 1 - y_prob[idx]
    
    ax.imshow(img, cmap='gray')
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f"True: {['No Crack', 'Crack'][true_label]}\nPred: {['No Crack', 'Crack'][pred_label]}\nConf: {confidence:.2f}", 
                 color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle("Sample Test Predictions (Green=Correct, Red=Wrong)", fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

## 9. Save Model & Results

Export trained model and predictions.

In [ ]:
import os

# Create output directories
os.makedirs('outputs/models', exist_ok=True)
os.makedirs('outputs/predictions', exist_ok=True)

# Save model
torch.save(model.state_dict(), 'outputs/models/impedance_cnn_best.pt')
print("✅ Model saved to outputs/models/impedance_cnn_best.pt")

# Save predictions
results_df = pd.DataFrame({
    'true_label': y_true,
    'predicted_label': y_pred,
    'crack_probability': y_prob,
    'correct': y_true == y_pred
})
results_df.to_csv('outputs/predictions/test_predictions.csv', index=False)
print("✅ Predictions saved to outputs/predictions/test_predictions.csv")

# Save metrics summary
metrics_summary = {
    'architecture': ARCH,
    'epochs': EPOCHS,
    'test_accuracy': float(test_acc),
    'test_loss': float(test_loss),
    'roc_auc': float(roc_auc),
    'confusion_matrix': cm.tolist(),
    'best_val_acc': float(best_val_acc)
}

import json
with open('outputs/metrics_summary.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)
print("✅ Metrics summary saved to outputs/metrics_summary.json")

print("\n" + "="*60)
print("TRAINING & EVALUATION COMPLETE")
print("="*60)
print(f"Model Architecture: {ARCH}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print("="*60)